## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [8]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


all dependencies present
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 5 — Class Imbalance Handling

Compares four ways of addressing the 7–10% positive rate, on one model
family, with everything inside the fold.

## What changed from R01

| Change | Reason |
|---|---|
| Resampling fitted **inside the training fold**, after in-fold preprocessing | M8 claimed this; now the code enforces it |
| No test-set scoring | GATE-1(iii) |
| Explicit verification that no synthetic row reaches a validation fold | R1 asserts "100% real data" in every evaluation partition — this makes it checkable |
| AUC-PR primary | matches M15 |
| Reweighting compared as a *fourth* option | Class weighting is the mechanism E-LightGBM's focal α implements, so it belongs in this comparison, not only in the ablation |

## Why this matters for GATE-2

M8 says the proposed model used no augmentation, and the focal objective
handles imbalance instead. Fine. But `is_unbalance=True` is honoured only by
LightGBM's built-in objectives and is **inert under a custom objective**, so
the R01 baseline received an explicit weighting correction that the proposed
arm never got. The comparison below establishes how much that correction is
worth on its own, which is what makes the ablation grid in Notebook 6b
interpretable.

In [9]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      score_binary, two_level_variance)
from losses import balanced_weights

banner("NOTEBOOK 5 — CLASS IMBALANCE")
OUT = run_dir("notebook05_imbalance")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
IMB_SEEDS = SEEDS[:3]
print(f"train_pool {train_pool.shape}  seeds {IMB_SEEDS}  |  test untouched")

try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler
    HAVE_IMB = True
except ImportError:
    HAVE_IMB = False
    print("imbalanced-learn unavailable — install it or state the omission in M8")

NOTEBOOK 5 — CLASS IMBALANCE
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records   (PRIMARY — school records only)
primary metric  : auc_pr
train_pool (784, 42)  seeds [42, 123, 456]  |  test untouched


In [11]:
# ---- strategies ---------------------------------------------------------
# Each takes (X_tr, y_tr) and returns (X, y, sample_weight). Only the
# TRAINING fold is ever modified; the validation fold is returned untouched.
def s_none(X, y):
    return X, y, None

def s_weight(X, y):
    return X, y, balanced_weights(y)

def s_smote(X, y):
    n_pos = int((y == 1).sum())
    k = max(1, min(5, n_pos - 1))          # k_neighbors must be < n_minority
    Xr, yr = SMOTE(random_state=SPLIT_SEED, k_neighbors=k).fit_resample(X, y)
    return pd.DataFrame(Xr, columns=X.columns), pd.Series(yr), None

def s_ros(X, y):
    Xr, yr = RandomOverSampler(random_state=SPLIT_SEED).fit_resample(X, y)
    return pd.DataFrame(Xr, columns=X.columns), pd.Series(yr), None

def s_rus(X, y):
    Xr, yr = RandomUnderSampler(random_state=SPLIT_SEED).fit_resample(X, y)
    return pd.DataFrame(Xr, columns=X.columns), pd.Series(yr), None

STRATEGIES = {"none": s_none, "class_weight": s_weight}
if HAVE_IMB:
    STRATEGIES.update({"SMOTE": s_smote, "RandomOverSample": s_ros,
                       "RandomUnderSample": s_rus})
print("strategies:", list(STRATEGIES))

strategies: ['none', 'class_weight', 'SMOTE', 'RandomOverSample', 'RandomUnderSample']


In [12]:
# ---- run ----------------------------------------------------------------
from lightgbm import LGBMClassifier
import time

rows, integrity = [], []
t0 = time.perf_counter()
for seed in IMB_SEEDS:
    for fi, (tr, vl) in enumerate(cv_splits(train_pool, seed), 1):
        X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
            train_pool.iloc[tr], train_pool.iloc[vl])
        n_val_before = len(y_vl)
        for name, fn in STRATEGIES.items():
            Xa, ya, sw = fn(X_tr, y_tr)
            model = LGBMClassifier(objective="binary", random_state=seed,
                                   **SHARED_PARAMS)
            model.fit(Xa, ya, sample_weight=sw)
            p = model.predict_proba(X_vl)[:, 1]
            rows.append({"seed": seed, "fold": fi, "arm": name,
                         "n_train_rows": len(ya),
                         "n_train_positive": int((ya == 1).sum()),
                         "train_positive_pct": round(100*float((ya==1).mean()), 1),
                         **score_binary(y_vl, p)})
            integrity.append({"seed": seed, "fold": fi, "strategy": name,
                              "val_rows_unchanged": len(y_vl) == n_val_before,
                              "val_rows": len(y_vl)})
    print(f"  seed {seed} [{time.perf_counter()-t0:.0f}s]")

fold_df = pd.DataFrame(rows)
fold_df.to_csv(OUT / "imbalance_fold_scores.csv", index=False)

# R1's claim that every evaluation partition is 100% real, made checkable
ig = pd.DataFrame(integrity)
ig.to_csv(OUT / "validation_fold_integrity.csv", index=False)
assert ig["val_rows_unchanged"].all(), "a strategy altered a validation fold"
print(f"\nINTEGRITY: all {len(ig)} (strategy, fold) pairs left the validation "
      "fold untouched. No synthetic or resampled row entered any evaluation "
      "partition. This is the check behind R1's '100% real data' sentence.")

  seed 42 [36s]
  seed 123 [71s]
  seed 456 [104s]

INTEGRITY: all 375 (strategy, fold) pairs left the validation fold untouched. No synthetic or resampled row entered any evaluation partition. This is the check behind R1's '100% real data' sentence.


In [13]:
# ---- results ------------------------------------------------------------
var = two_level_variance(fold_df, PRIMARY_METRIC)
summary = (fold_df.groupby("arm")
           .agg(auc_pr=("auc_pr", "mean"), auc_roc=("auc_roc", "mean"),
                recall=("recall", "mean"), precision=("precision", "mean"),
                train_rows=("n_train_rows", "mean"),
                train_pos_pct=("train_positive_pct", "mean"))
           .reset_index()
           .merge(var[["arm", "between_seed_sd", "mean_within_seed_fold_sd"]],
                  on="arm")
           .sort_values("auc_pr", ascending=False))
summary.to_csv(OUT / "imbalance_summary.csv", index=False)
print(summary.round(4).to_string(index=False))

base = summary[summary["arm"] == "none"]["auc_pr"].iloc[0]
print(f"\nrelative to no handling ({base:.4f}):")
for _, r in summary.iterrows():
    if r["arm"] != "none":
        print(f"  {r['arm']:22s} {r['auc_pr']-base:+.4f}  "
              f"(between-seed SD {r['between_seed_sd']:.4f})")
print("\nRead the deltas against the SD, not against zero. Any delta smaller "
      "than its own between-seed SD is not a finding.")

cw = summary[summary["arm"] == "class_weight"]["auc_pr"]
if len(cw):
    print(f"\nclass weighting alone is worth {float(cw.iloc[0])-base:+.4f} AUC-PR. "
          "That is the size of the correction the R01 baseline received and the "
          "proposed arm did not — the asymmetry GATE-2 turns on.")

write_manifest(OUT, {"notebook": "05_imbalance", "test_set_scored": False,
                     "seeds": IMB_SEEDS, "strategies": list(STRATEGIES),
                     "validation_integrity_all_pass": True})

              arm  auc_pr  auc_roc  recall  precision  train_rows  train_pos_pct  between_seed_sd  mean_within_seed_fold_sd
             none  0.9812   0.9961  0.9045     0.9688       627.2           8.82           0.0035                    0.0245
     class_weight  0.9803   0.9959  0.9169     0.9632       627.2           8.82           0.0037                    0.0247
 RandomOverSample  0.9786   0.9949  0.8944     0.9400      1144.0          50.00           0.0010                    0.0265
            SMOTE  0.9771   0.9937  0.9266     0.9555      1144.0          50.00           0.0025                    0.0292
RandomUnderSample  0.9559   0.9892  0.9575     0.7292       110.4          50.00           0.0085                    0.0418

relative to no handling (0.9812):
  class_weight           -0.0008  (between-seed SD 0.0037)
  RandomOverSample       -0.0026  (between-seed SD 0.0010)
  SMOTE                  -0.0040  (between-seed SD 0.0025)
  RandomUnderSample      -0.0253  (between-s

{'run_dir': 'results/notebook05_imbalance/20260921T143133Z_records',
 'generated_utc': '2026-09-21T14:33:24Z',
 'git': {'commit': '',
  'branch': '',
  'dirty': False,
  'dirty_paths': [],
  'no_git': True,
  'freeze_tag': ''},
 'host': '478503693bde',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39',
 'python': '3.13.15',
 'protocol': {'split_seed': 42,
  'test_size': 0.2,
  'seeds': [42, 123, 456, 789, 1024, 2048, 3333, 5555, 7777, 9999],
  'n_splits': 5,
  'n_repeats': 5,
  'primary_metric': 'auc_pr',
  'shared_params': {'n_estimators': 300,
   'num_leaves': 31,
   'learning_rate': 0.05,
   'subsample': 0.8,
   'colsample_bytree': 0.8,
   'verbosity': -1},
  'gamma_reported': 2.0,
  'alpha_reported': 0.75,
  'school_handling': 'drop',
  'feature_set': 'records',
  'active_composites': ['attendance_risk_index'],
  'fairness_threshold': 0.1,
  'drop_derived_duplicates': True,
  'suspect_column_actions': {'social_studies_exam_score': 'keep'}},
 'notebook': '05_imbalance',
 'test_set

---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [14]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook05_imbalance/20260921T143133Z_records
files written : 6
    RUN_MANIFEST.json
    environment_versions.csv
    imbalance_fold_scores.csv
    imbalance_summary.csv
    pip_freeze.txt
    validation_fold_integrity.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, a